# Module 26: Sales Forecasting Case Study
In this case study, I will build a predictive model to forecast sales using the provided Superstore Sales dataset. I'll clean the data, extract time features from the dates, and use a Linear Regression model to make predictions!

### Step 1: Import and Explore the Dataset
First, let's load pandas and our dataset to see what we are working with.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('train.csv')

# Let's check the first few rows to understand the data
df.head()

In [ ]:
# Let's inspect the data types and check for missing values
print(df.info())
print("\nMissing values:\n", df.isnull().sum())

### Step 2: Data Preprocessing
The dataset has some missing values in `Postal Code`, but since we are forecasting total `Sales` over time, we don't really need the postal code right now. 

More importantly, the `Order Date` is just a string (object). We need to convert it to a proper datetime format so Python knows it represents time. Then, we will aggregate our sales by day to predict daily sales trends.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 1. Convert 'Order Date' to datetime format. 
# Note: The dates in the CSV are in DD/MM/YYYY format
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')

# 2. We only care about the date and the sales for forecasting
sales_data = df[['Order Date', 'Sales']]

# 3. Group by date to get the total daily sales
daily_sales = sales_data.groupby('Order Date').sum().reset_index()

# Sort the values by date just to be safe
daily_sales = daily_sales.sort_values('Order Date')

daily_sales.head()

In [ ]:
# Let's plot our daily sales to see the trend!
plt.figure(figsize=(15, 6))
plt.plot(daily_sales['Order Date'], daily_sales['Sales'])
plt.title('Total Daily Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.show()

In [ ]:
# 4. Extract relevant features for our machine learning model
# A regression model needs numbers, not dates, so we extract Year, Month, and Day
daily_sales['Year'] = daily_sales['Order Date'].dt.year
daily_sales['Month'] = daily_sales['Order Date'].dt.month
daily_sales['Day'] = daily_sales['Order Date'].dt.day

# We can drop the original datetime column now since we extracted the features
X = daily_sales[['Year', 'Month', 'Day']]
y = daily_sales['Sales']

X.head()

### Step 3: Split the Dataset into Training and Testing Sets
Because this is time series data, we usually don't shuffle it randomly. Instead, we train on the past and test on the future. Let's use the first 80% of days for training and the last 20% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

# Split data. Note shuffle=False because time order matters!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

print("Training days:", len(X_train))
print("Testing days:", len(X_test))

### Step 4: Apply a Forecasting Model
I will use a simple `Linear Regression` model. It will try to find a mathematical relationship between the date features (Year, Month, Day) and the Sales amount.

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize the model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Predict the sales for our testing dates
predictions = model.predict(X_test)

### Step 5: Evaluate the Model Performance
Let's check how far off our predictions were using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² score.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

# Let's visualize the actual vs predicted sales for the test set
plt.figure(figsize=(15, 6))
plt.plot(y_test.values, label='Actual Sales', alpha=0.7)
plt.plot(predictions, label='Predicted Sales', color='red', alpha=0.7)
plt.title('Actual vs Predicted Sales (Test Data)')
plt.legend()
plt.show()

### Step 6: Analysis and Findings

**My Observations:**
1. **Data Volatility:** As seen in the first chart, the daily sales are highly volatile. There are massive spikes on certain days and very low sales on others. This makes predicting the exact daily sales very difficult for a basic model.
2. **Model Performance:** The Linear Regression model learned a general baseline trend, but the R² score is quite low (close to zero or negative). This means that a simple straight line based just on the year, month, and day is not complex enough to capture the wild daily fluctuations in sales.
3. **Visual Proof:** The final visualization shows that the predicted line (red) is relatively flat compared to the massive spikes of the actual sales (blue). Linear Regression captures the average trend but completely misses the sudden spikes.
4. **Future Improvements:** To get a better forecast, we should try a model specifically built for Time Series like **ARIMA**, **Prophet**, or an advanced tree-based model like Random Forest. Grouping the data into **Monthly Sales** instead of Daily Sales would also significantly reduce the noise and make forecasting much more accurate!